# Larme 7B — QLoRA fine-tune on a free Colab T4

Runtime → Change runtime type → T4 GPU. Then run the cells top to bottom.
It fine-tunes an open 7B base with 4-bit NF4 + LoRA on the 417-pair instruction set,
and writes an adapter to `/content/adapter`.

This does **not** train 7B from scratch — that needs ~13,000 A100-hours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" datasets accelerate

In [ ]:
# Upload pairs.jsonl (417 instruction pairs) when prompted.
from google.colab import files
up = files.upload()
print(list(up))

In [ ]:
# Paste the training script from the repository:
# https://github.com/aniso199191/pictureimages-light-series-ai/blob/main/kaggle/qlora_7b.py
!wget -q -O qlora_7b.py https://raw.githubusercontent.com/aniso199191/pictureimages-light-series-ai/main/kaggle/qlora_7b.py
!python qlora_7b.py --data pairs.jsonl --out /content/adapter --epochs 2

In [ ]:
# Package the adapter and download it (~150-250 MB).
!zip -qr adapter.zip /content/adapter
from google.colab import files
files.download('/content/adapter.zip')

## Optional: merge into full 7B weights (~14 GB)

Only if you have the disk and the RAM. GitHub rejects files over 2 GB,
so split the result or publish it on GitOpen.

In [ ]:
!python qlora_7b.py --data pairs.jsonl --out /content/adapter --epochs 2 --merge
!ls -lh /content/adapter-merged | head